#  Reto Semana 8: MeteoSense Analytics

## Sistema de Análisis de Datos Meteorológicos con NumPy

---

###  Contexto del Proyecto

**MeteoSense** es una startup que instala redes de sensores meteorológicos en ciudades de México para monitorear condiciones ambientales.

###  Objetivos de Aprendizaje

En este reto aplicarás:
- Creación y manipulación de arrays NumPy
- Indexación y slicing para acceso a datos
- Operaciones vectorizadas para cálculos eficientes
- Funciones estadísticas de NumPy
- Manejo de valores faltantes (NaN)
- Broadcasting para operaciones entre arrays

##  Configuración Inicial

In [1]:
import numpy as np

# Configuración para reproducibilidad
np.random.seed(42)

print(' NumPy cargado correctamente')
print(f'   Versión: {np.__version__}')

 NumPy cargado correctamente
   Versión: 2.4.4


##  Datos del Reto

Ejecuta la siguiente celda para generar los datos de los sensores.

In [2]:
estaciones = ['Coyoacán', 'Azcapotzalco', 'Xochimilco', 'Tlalpan', 'Miguel Hidalgo']
n_estaciones = len(estaciones)
n_dias = 7
n_horas = 24

temp_base = np.array([22, 24, 20, 19, 23])
hora_del_dia = np.arange(24)
variacion_diaria = 5 * np.sin((hora_del_dia - 6) * np.pi / 12)

temperatura = np.zeros((n_estaciones, n_dias, n_horas))
for i in range(n_estaciones):
    for d in range(n_dias):
        temperatura[i, d, :] = temp_base[i] + variacion_diaria + np.random.normal(0, 1.5, n_horas)

temperatura[1, 2, 10:14] = np.nan
temperatura[3, 5, 0:3]   = np.nan

humedad_base = np.array([55, 45, 70, 65, 50])
variacion_humedad = -15 * np.sin((hora_del_dia - 6) * np.pi / 12)

humedad = np.zeros((n_estaciones, n_dias, n_horas))
for i in range(n_estaciones):
    for d in range(n_dias):
        humedad[i, d, :] = humedad_base[i] + variacion_humedad + np.random.normal(0, 5, n_horas)

humedad = np.clip(humedad, 20, 95)
humedad[0, 4, 15:18] = np.nan

co2_base = np.array([380, 420, 360, 350, 410])
patron_trafico = np.zeros(24)
patron_trafico[7:10]  = 30
patron_trafico[17:20] = 40
patron_trafico[12:14] = 15

co2 = np.zeros((n_estaciones, n_dias, n_horas))
for i in range(n_estaciones):
    for d in range(n_dias):
        co2[i, d, :] = co2_base[i] + patron_trafico + np.random.normal(0, 10, n_horas)

co2[:, 3, :] *= 1.15
co2[2, 1, 5:8] = np.nan

temp_promedio_diario    = np.nanmean(temperatura, axis=2)
humedad_promedio_diario = np.nanmean(humedad, axis=2)
co2_promedio_diario     = np.nanmean(co2, axis=2)

print('Datos generados exitosamente')
print(f'temperatura shape: {temperatura.shape}')
print(f'Estaciones: {estaciones}')

Datos generados exitosamente
temperatura shape: (5, 7, 24)
Estaciones: ['Coyoacán', 'Azcapotzalco', 'Xochimilco', 'Tlalpan', 'Miguel Hidalgo']


---
##  PARTE 1: Exploración de Arrays (25 puntos)
### Ejercicio 1.1: Inspección de Datos (5 puntos)

In [3]:
# 
#                    EJERCICIO 1.1: INSPECCIÓN
# 

# 1. Número de dimensiones del array temperatura
n_dimensiones = temperatura.ndim          # ndim devuelve el número de ejes

# 2. Forma (shape) del array
forma = temperatura.shape                 # shape → (estaciones, días, horas)

# 3. Número total de elementos
total_elementos = temperatura.size        # size = 5 × 7 × 24 = 840

# 4. Tipo de datos
tipo_datos = temperatura.dtype            # float64 por defecto

# 5. Tamaño en memoria (bytes)
memoria_bytes = temperatura.nbytes        # size × bytes_por_elemento

print(' PROPIEDADES DEL ARRAY TEMPERATURA')
print('' * 40)
print(f'Dimensiones: {n_dimensiones}D')
print(f'Forma: {forma}')
print(f'  → {forma[0]} estaciones')
print(f'  → {forma[1]} días')
print(f'  → {forma[2]} horas por día')
print(f'Total de mediciones: {total_elementos:,}')
print(f'Tipo de datos: {tipo_datos}')
print(f'Memoria: {memoria_bytes:,} bytes ({memoria_bytes/1024:.2f} KB)')

 PROPIEDADES DEL ARRAY TEMPERATURA

Dimensiones: 3D
Forma: (5, 7, 24)
  → 5 estaciones
  → 7 días
  → 24 horas por día
Total de mediciones: 840
Tipo de datos: float64
Memoria: 6,720 bytes (6.56 KB)


### Ejercicio 1.2: Indexación Básica (10 puntos)

In [4]:
# 
#                    EJERCICIO 1.2: INDEXACIÓN
# 

# 1. Temperatura de Coyoacán (índice 0), día 1 (índice 0), a las 12:00 (índice 12)
temp_coyoacan_d1_12h = temperatura[0, 0, 12]   # [estación, día, hora]
print(f' Coyoacán, Día 1, 12:00h: {temp_coyoacan_d1_12h:.1f}°C')

# 2. Todas las temperaturas de Xochimilco (índice 2) en el día 3 (índice 2)
#    : en el eje de horas devuelve las 24 lecturas
temp_xochimilco_d3 = temperatura[2, 2, :]       # shape → (24,)
print(f'\n Xochimilco, Día 3 (24 horas):')
print(f'   Primeras 6 horas: {temp_xochimilco_d3[:6].round(1)}')

# 3. Temperatura promedio diario de Miguel Hidalgo (índice 4) para los 7 días
#    temp_promedio_diario tiene shape (5, 7); fila 4 = Miguel Hidalgo
temp_mh_7dias = temp_promedio_diario[4]          # shape → (7,)
print(f'\n Miguel Hidalgo - Promedio por día:')
print(f'   {temp_mh_7dias.round(1)}')

# 4. Último valor de CO2 registrado — índice -1 selecciona el último elemento
ultimo_co2 = co2[-1, -1, -1]                    # última estación, último día, última hora
print(f'\n Último CO2 registrado: {ultimo_co2:.1f} ppm')

 Coyoacán, Día 1, 12:00h: 27.4°C

 Xochimilco, Día 3 (24 horas):
   Primeras 6 horas: [13.9 15.4 16.2 19.3 18.9 17.8]

 Miguel Hidalgo - Promedio por día:
   [23.  22.9 22.8 23.1 23.2 23.  23.2]

 Último CO2 registrado: 401.5 ppm


### Ejercicio 1.3: Slicing Avanzado (10 puntos)

In [5]:
# 
#                    EJERCICIO 1.3: SLICING
# 

# 1. Temperaturas de TODAS las estaciones, TODOS los días, solo horas de la TARDE (12-17)
#    Las horas 12 a 17 corresponden a los índices 12:18 (el límite superior es exclusivo)
temp_tardes = temperatura[:, :, 12:18]          # shape → (5, 7, 6)
print(f' Temperaturas de tardes (12-17h)')
print(f'   Shape: {temp_tardes.shape}')

# 2. Humedad de las primeras 3 estaciones, últimos 3 días, todas las horas
#    :3 selecciona índices 0,1,2  |  -3: selecciona los últimos 3
humedad_subset = humedad[:3, -3:, :]            # shape → (3, 3, 24)
print(f'\n Subset de humedad')
print(f'   Shape: {humedad_subset.shape}')

# 3. CO2 de estaciones pares (0, 2, 4), todos los días, horas de mañana (6-11)
#    ::2 en el eje 0 selecciona índices 0, 2, 4
co2_mañanas_pares = co2[::2, :, 6:12]          # shape → (3, 7, 6)
print(f'\n CO2 mañanas (estaciones pares)')
print(f'   Shape: {co2_mañanas_pares.shape}')

# 4. Temperaturas en orden inverso de días
#    ::-1 recorre el eje de días de atrás hacia adelante
temp_inverso = temperatura[:, ::-1, :]          # shape → (5, 7, 24)
print(f'\n Temperatura días invertidos')
print(f'   Shape: {temp_inverso.shape}')

 Temperaturas de tardes (12-17h)
   Shape: (5, 7, 6)

 Subset de humedad
   Shape: (3, 3, 24)

 CO2 mañanas (estaciones pares)
   Shape: (3, 7, 6)

 Temperatura días invertidos
   Shape: (5, 7, 24)


---
##  PARTE 2: Estadísticas Básicas (25 puntos)
### Ejercicio 2.1: Estadísticas Globales (10 puntos)

In [6]:
# 
#                 EJERCICIO 2.1: ESTADÍSTICAS GLOBALES
# 
# Se usan funciones nan* para ignorar los NaN al calcular

# 1. Temperatura promedio global
temp_promedio = np.nanmean(temperatura)

# 2. Temperatura máxima registrada
temp_maxima = np.nanmax(temperatura)

# 3. Temperatura mínima registrada
temp_minima = np.nanmin(temperatura)

# 4. Desviación estándar de temperatura
temp_std = np.nanstd(temperatura)

# 5. Rango de temperatura (máxima - mínima)
temp_rango = temp_maxima - temp_minima

print('')
print('           ESTADÍSTICAS GLOBALES DE TEMPERATURA               ')
print('')
print(f'  Promedio:     {temp_promedio:>6.2f} °C                              ')
print(f'  Máxima:       {temp_maxima:>6.2f} °C                              ')
print(f'  Mínima:       {temp_minima:>6.2f} °C                              ')
print(f'  Desv. Est.:   {temp_std:>6.2f} °C                              ')
print(f'  Rango:        {temp_rango:>6.2f} °C                              ')
print('')


           ESTADÍSTICAS GLOBALES DE TEMPERATURA               

  Promedio:      21.59 °C                              
  Máxima:        32.91 °C                              
  Mínima:        10.62 °C                              
  Desv. Est.:     4.28 °C                              
  Rango:         22.29 °C                              



### Ejercicio 2.2: Estadísticas por Eje (15 puntos)

In [7]:
# 
#                 EJERCICIO 2.2: ESTADÍSTICAS POR EJE
# 

# 1. Temperatura promedio POR ESTACIÓN
#    Promediamos sobre los ejes 1 (días) y 2 (horas) → queda eje 0 (estaciones)
#    Resultado: array de 5 elementos
temp_por_estacion = np.nanmean(temperatura, axis=(1, 2))

print(' TEMPERATURA PROMEDIO POR ESTACIÓN')
print('' * 40)
for i, est in enumerate(estaciones):
    print(f'   {est:15s}: {temp_por_estacion[i]:5.1f} °C')

# 2. Humedad promedio POR HORA DEL DÍA
#    Promediamos sobre los ejes 0 (estaciones) y 1 (días) → queda eje 2 (horas)
#    Resultado: array de 24 elementos
humedad_por_hora = np.nanmean(humedad, axis=(0, 1))

print('\n HUMEDAD PROMEDIO POR HORA')
print('' * 40)
print('   Hora  Humedad')
for h in [0, 6, 12, 18]:
    print(f'   {h:02d}:00  {humedad_por_hora[h]:5.1f}%')

# 3. CO2 máximo POR DÍA
#    Máximo sobre los ejes 0 (estaciones) y 2 (horas) → queda eje 1 (días)
#    Resultado: array de 7 elementos
co2_max_por_dia = np.nanmax(co2, axis=(0, 2))

print('\n CO2 MÁXIMO POR DÍA')
print('' * 40)
for d in range(n_dias):
    print(f'   Día {d+1}: {co2_max_por_dia[d]:6.1f} ppm')

 TEMPERATURA PROMEDIO POR ESTACIÓN

   Coyoacán       :  21.9 °C
   Azcapotzalco   :  24.0 °C
   Xochimilco     :  20.0 °C
   Tlalpan        :  19.0 °C
   Miguel Hidalgo :  23.0 °C

 HUMEDAD PROMEDIO POR HORA

   Hora  Humedad
   00:00   72.1%
   06:00   58.2%
   12:00   43.7%
   18:00   57.1%

 CO2 MÁXIMO POR DÍA

   Día 1:  471.3 ppm
   Día 2:  466.2 ppm
   Día 3:  465.0 ppm
   Día 4:  541.5 ppm
   Día 5:  466.5 ppm
   Día 6:  467.3 ppm
   Día 7:  469.4 ppm


---
##  PARTE 3: Operaciones Vectorizadas (25 puntos)
### Ejercicio 3.1: Conversiones de Unidades (10 puntos)

In [8]:
# 
#              EJERCICIO 3.1: CONVERSIONES VECTORIZADAS
# 
# NumPy aplica la operación a cada elemento sin necesidad de loops

# 1. Convertir temperatura de Celsius a Fahrenheit
#    F = C × 9/5 + 32
temperatura_fahrenheit = temperatura * 9/5 + 32

print(' TEMPERATURA EN FAHRENHEIT')
print(f'   Promedio: {np.nanmean(temperatura_fahrenheit):.1f} °F')
print(f'   Máxima:   {np.nanmax(temperatura_fahrenheit):.1f} °F')
print(f'   Mínima:   {np.nanmin(temperatura_fahrenheit):.1f} °F')

# 2. Convertir temperatura de Celsius a Kelvin
#    K = C + 273.15
temperatura_kelvin = temperatura + 273.15

print(f'\n TEMPERATURA EN KELVIN')
print(f'   Promedio: {np.nanmean(temperatura_kelvin):.1f} K')

# 3. Normalizar humedad a rango [0, 1]
#    Fórmula Min-Max: (valor - min) / (max - min)
humedad_min = np.nanmin(humedad)
humedad_max = np.nanmax(humedad)
humedad_normalizada = (humedad - humedad_min) / (humedad_max - humedad_min)

print(f'\n HUMEDAD NORMALIZADA [0-1]')
print(f'   Promedio: {np.nanmean(humedad_normalizada):.3f}')
print(f'   Min:      {np.nanmin(humedad_normalizada):.3f}')
print(f'   Max:      {np.nanmax(humedad_normalizada):.3f}')

 TEMPERATURA EN FAHRENHEIT
   Promedio: 70.9 °F
   Máxima:   91.2 °F
   Mínima:   51.1 °F

 TEMPERATURA EN KELVIN
   Promedio: 294.7 K

 HUMEDAD NORMALIZADA [0-1]
   Promedio: 0.510
   Min:      0.000
   Max:      1.000


### Ejercicio 3.2: Índice de Confort Térmico (15 puntos)

```
ICT = T + 0.05 × H     (T en °C, H en %)
ICT < 20      → Frío
20 ≤ ICT < 25 → Confortable
25 ≤ ICT < 30 → Cálido
ICT ≥ 30      → Muy caluroso
```

In [9]:
# 
#              EJERCICIO 3.2: ÍNDICE DE CONFORT TÉRMICO
# 

# 1. ICT para TODAS las mediciones — operación vectorizada elemento a elemento
#    Ambos arrays tienen shape (5, 7, 24) → el resultado también
ict = temperatura + 0.05 * humedad

print(' ÍNDICE DE CONFORT TÉRMICO (ICT)')
print('' * 45)
print(f'   Shape del array ICT: {ict.shape}')
print(f'   ICT promedio: {np.nanmean(ict):.2f}')
print(f'   ICT máximo:   {np.nanmax(ict):.2f}')
print(f'   ICT mínimo:   {np.nanmin(ict):.2f}')

# 2. Clasificar condiciones con indexación booleana
#    np.isnan(ict) produce True donde hay NaN; ~ invierte la máscara
#    ict[~np.isnan(ict)] extrae solo valores válidos para no contaminar los conteos
ict_valido = ict[~np.isnan(ict)]

n_frio         = np.sum(ict_valido < 20)
n_confortable  = np.sum((ict_valido >= 20) & (ict_valido < 25))
n_calido       = np.sum((ict_valido >= 25) & (ict_valido < 30))
n_muy_caluroso = np.sum(ict_valido >= 30)

n_validas = np.sum(~np.isnan(ict))

print('\n DISTRIBUCIÓN DE CONDICIONES')
print('' * 45)
print(f'     Frío (<20):           {n_frio:5d} ({100*n_frio/n_validas:5.1f}%)')
print(f'    Confortable (20-25):  {n_confortable:5d} ({100*n_confortable/n_validas:5.1f}%)')
print(f'     Cálido (25-30):       {n_calido:5d} ({100*n_calido/n_validas:5.1f}%)')
print(f'    Muy caluroso (≥30):   {n_muy_caluroso:5d} ({100*n_muy_caluroso/n_validas:5.1f}%)')
print(f'   ')
print(f'   Total válidas:           {n_validas:5d}')

 ÍNDICE DE CONFORT TÉRMICO (ICT)

   Shape del array ICT: (5, 7, 24)
   ICT promedio: 24.45
   ICT máximo:   34.54
   ICT mínimo:   14.17

 DISTRIBUCIÓN DE CONDICIONES

     Frío (<20):             103 ( 12.4%)
    Confortable (20-25):    353 ( 42.5%)
     Cálido (25-30):         325 ( 39.2%)
    Muy caluroso (≥30):      49 (  5.9%)
   
   Total válidas:             830


---
##  PARTE 4: Análisis Avanzado (25 puntos)
### Ejercicio 4.1: Detección de Anomalías (10 puntos)

In [10]:
# 
#              EJERCICIO 4.1: DETECCIÓN DE ANOMALÍAS
# 
# Criterio: anómalo si |valor - media| > 2 * desviación_estándar

# 1. Media y desviación estándar del CO2 (ignorando NaN)
co2_media = np.nanmean(co2)
co2_std   = np.nanstd(co2)

# 2. Límites de la zona normal (media ± 2σ)
limite_inferior = co2_media - 2 * co2_std
limite_superior = co2_media + 2 * co2_std

print(' ANÁLISIS DE ANOMALÍAS EN CO2')
print('' * 45)
print(f'   Media CO2:       {co2_media:.1f} ppm')
print(f'   Desv. Est.:      {co2_std:.1f} ppm')
print(f'   Límite inferior: {limite_inferior:.1f} ppm')
print(f'   Límite superior: {limite_superior:.1f} ppm')

# 3. Máscara booleana: True donde el valor existe Y está fuera del rango
#    ~np.isnan(co2)  → excluye NaN (no podemos comparar con NaN)
#    La condición de anomalía es OR entre por debajo y por encima del rango
mascara_anomalias = (~np.isnan(co2)) & ((co2 < limite_inferior) | (co2 > limite_superior))

# 4. Número de anomalías
n_anomalias = np.sum(mascara_anomalias)

# 5. Valores anómalos
valores_anomalos = co2[mascara_anomalias]

print(f'\n  ANOMALÍAS DETECTADAS: {n_anomalias}')
if n_anomalias > 0:
    print(f'   Valores: {valores_anomalos[:10].round(1)}')
    if n_anomalias > 10:
        print(f'   ... y {n_anomalias - 10} más')

 ANÁLISIS DE ANOMALÍAS EN CO2

   Media CO2:       402.6 ppm
   Desv. Est.:      39.6 ppm
   Límite inferior: 323.4 ppm
   Límite superior: 481.7 ppm

  ANOMALÍAS DETECTADAS: 31
   Valores: [486.6 490.4 489.3 483.5 502.5 491.6 485.8 506.9 514.2 530.2]
   ... y 21 más


### Ejercicio 4.2: Análisis de Contingencia Ambiental (15 puntos)

In [11]:
# 
#           EJERCICIO 4.2: ANÁLISIS DE CONTINGENCIA AMBIENTAL
# 

DIA_CONTINGENCIA = 3  # índice del día 4

# 1. Datos de CO2 del día de contingencia → todas las estaciones y horas
co2_contingencia = co2[:, DIA_CONTINGENCIA, :]   # shape → (5, 24)

# 2. Datos de CO2 de los días normales
dias_normales    = [0, 1, 2, 4, 5, 6]
co2_dias_normales = co2[:, dias_normales, :]      # shape → (5, 6, 24)

# 3. Promedio CO2 en el día de contingencia
promedio_contingencia = np.nanmean(co2_contingencia)

# 4. Promedio CO2 en días normales
promedio_normal = np.nanmean(co2_dias_normales)

# 5. Incremento porcentual
incremento_porcentual = ((promedio_contingencia - promedio_normal) / promedio_normal) * 100

print('')
print('           ANÁLISIS DE CONTINGENCIA AMBIENTAL                 ')
print('                        Día 4                                 ')
print('')
print(f'  CO2 promedio día contingencia: {promedio_contingencia:>7.1f} ppm              ')
print(f'  CO2 promedio días normales:    {promedio_normal:>7.1f} ppm              ')
print(f'  Incremento:                    {incremento_porcentual:>7.1f} %               ')
print('')

# 6. Estación más afectada
#    Promedio por estación en contingencia: axis=1 (promedia las 24 horas)
co2_por_estacion_contingencia = np.nanmean(co2_contingencia, axis=1)    # shape → (5,)
#    Promedio por estación en días normales: axis=(1,2) (promedia días y horas)
co2_por_estacion_normal       = np.nanmean(co2_dias_normales, axis=(1, 2))  # shape → (5,)

incremento_por_estacion = ((co2_por_estacion_contingencia - co2_por_estacion_normal) /
                            co2_por_estacion_normal) * 100

# argmax devuelve el índice del valor más alto
idx_mas_afectada = np.argmax(incremento_por_estacion)

print('\n IMPACTO POR ESTACIÓN')
print('' * 50)
for i, est in enumerate(estaciones):
    barra = '' * int(incremento_por_estacion[i] / 2)
    print(f'   {est:15s}: +{incremento_por_estacion[i]:5.1f}% {barra}')

print(f'\n  Estación más afectada: {estaciones[idx_mas_afectada]}')


           ANÁLISIS DE CONTINGENCIA AMBIENTAL                 
                        Día 4                                 

  CO2 promedio día contingencia:   453.4 ppm              
  CO2 promedio días normales:      394.1 ppm              
  Incremento:                       15.1 %               


 IMPACTO POR ESTACIÓN

   Coyoacán       : + 15.7% 
   Azcapotzalco   : + 15.0% 
   Xochimilco     : + 15.2% 
   Tlalpan        : + 14.3% 
   Miguel Hidalgo : + 15.2% 

  Estación más afectada: Coyoacán


---
##  EJERCICIO FINAL: Reporte Ejecutivo (BONUS - 10 puntos)

In [12]:
# 
#                    EJERCICIO BONUS: REPORTE EJECUTIVO
# 

# 1. Estación más calurosa — nanmean sobre días y horas, luego argmax
idx_mas_calurosa  = np.argmax(np.nanmean(temperatura, axis=(1, 2)))
estacion_mas_calurosa = estaciones[idx_mas_calurosa]

# 2. Estación más húmeda — misma lógica con humedad
idx_mas_humeda    = np.argmax(np.nanmean(humedad, axis=(1, 2)))
estacion_mas_humeda   = estaciones[idx_mas_humeda]

# 3. Estación con mejor calidad de aire — argmin sobre CO2 promedio
idx_mejor_aire    = np.argmin(np.nanmean(co2, axis=(1, 2)))
estacion_mejor_aire   = estaciones[idx_mejor_aire]

# 4. Hora más calurosa del día — promedio sobre estaciones (0) y días (1)
temp_por_hora      = np.nanmean(temperatura, axis=(0, 1))  # shape → (24,)
hora_mas_calurosa  = np.argmax(temp_por_hora)              # índice de la hora más alta

# 5. Hora con peor calidad de aire — promedio sobre estaciones (0) y días (1)
co2_por_hora       = np.nanmean(co2, axis=(0, 1))          # shape → (24,)
hora_peor_aire     = np.argmax(co2_por_hora)

# 6. Número de valores faltantes por variable
nan_temperatura = np.sum(np.isnan(temperatura))
nan_humedad     = np.sum(np.isnan(humedad))
nan_co2         = np.sum(np.isnan(co2))
total_nan       = nan_temperatura + nan_humedad + nan_co2

print('')
print('')
print('                                                                      ')
print('              METEOSENSE - REPORTE EJECUTIVO SEMANAL              ')
print('                        CDMX - Semana de Análisis                     ')
print('                                                                      ')
print('')
print('                                                                      ')
print('   RESUMEN DE CONDICIONES                                           ')
print('     ')
print(f'      Temperatura promedio:    {np.nanmean(temperatura):>5.1f} °C                        ')
print(f'     Humedad promedio:         {np.nanmean(humedad):>5.1f} %                         ')
print(f'     CO2 promedio:            {np.nanmean(co2):>6.1f} ppm                       ')
print('                                                                      ')
print('   RANKINGS                                                         ')
print('     ')
print(f'     Estación más calurosa:   {estacion_mas_calurosa:15s}                  ')
print(f'     Estación más húmeda:     {estacion_mas_humeda:15s}                  ')
print(f'     Mejor calidad de aire:   {estacion_mejor_aire:15s}                  ')
print('                                                                      ')
print('  ⏰ PATRONES TEMPORALES                                              ')
print('     ')
print(f'      Hora más calurosa:       {hora_mas_calurosa:02d}:00 hrs                          ')
print(f'     Hora con más CO2:         {hora_peor_aire:02d}:00 hrs                          ')
print('                                                                      ')
print('    CALIDAD DE DATOS                                                ')
print('     ')
print(f'    Valores faltantes totales:  {total_nan:4d}                                 ')
print(f'      - Temperatura: {nan_temperatura:3d}                                            ')
print(f'      - Humedad:     {nan_humedad:3d}                                            ')
print(f'      - CO2:         {nan_co2:3d}                                            ')
print('                                                                      ')
print('')
print('')



                                                                      
              METEOSENSE - REPORTE EJECUTIVO SEMANAL              
                        CDMX - Semana de Análisis                     
                                                                      

                                                                      
   RESUMEN DE CONDICIONES                                           
     
      Temperatura promedio:     21.6 °C                        
     Humedad promedio:          57.5 %                         
     CO2 promedio:             402.6 ppm                       
                                                                      
   RANKINGS                                                         
     
     Estación más calurosa:   Azcapotzalco                     
     Estación más húmeda:     Xochimilco                       
     Mejor calidad de aire:   Tlalpan                          
                                         